# Train the I-JEPA latent predictor

This config-driven notebook validates MVTec nominal splits, builds/reuses latent caches, trains the latent Transformer predictor with early stopping, and saves the best validation checkpoint. The official test split is not used for model selection.

In [ ]:
from pathlib import Path
import sys
import torch
import yaml
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
CONFIG_ROOT = PROJECT_ROOT / 'configs'
DATASET_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'mvtec_ad'
if not DATASET_ROOT.is_dir(): DATASET_ROOT = PROJECT_ROOT / 'dataset'
CATEGORIES = ['bottle', 'capsule', 'hazelnut']
USE_MOCK_ENCODER = False
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CHECKPOINT_PATH = PROJECT_ROOT / 'checkpoints' / 'IN22K-vit.g.16-600e.pth.tar'
OFFICIAL_REPO_PATH = PROJECT_ROOT / 'external' / 'ijepa'
CACHE_ROOT = PROJECT_ROOT / 'data' / 'cache' / 'latents'
OUTPUT_ROOT = PROJECT_ROOT / 'checkpoints' / 'predictors'
predictor_cfg = yaml.safe_load((CONFIG_ROOT / 'predictor' / 'base.yaml').read_text())
data_cfg = yaml.safe_load((CONFIG_ROOT / 'data' / 'mvtec.yaml').read_text())
architecture, training = predictor_cfg['architecture'], predictor_cfg['training']
# Nominal split settings come from configs/data/mvtec.yaml only (not the training seed).
VAL_FRACTION = float(data_cfg['nominal_split']['val_fraction'])
SPLIT_SEED = int(data_cfg['nominal_split']['seed'])
if USE_MOCK_ENCODER: DEVICE = 'cpu'
print(f'Root: {PROJECT_ROOT}\nDataset: {DATASET_ROOT}\nDevice: {DEVICE}')
if DEVICE == 'cuda': print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from ijepa_ad.data.mvtec_dataset import validate_category_splits
for category in CATEGORIES:
    train, test = validate_category_splits(DATASET_ROOT, category)
    print(f'{category:10s} train={len(train):4d} test={len(test):4d}')

## Build or reuse nominal latent caches

In [ ]:
from ijepa_ad.caching.cli import CacheRunConfig, run_cache_generation
cache_run = CacheRunConfig(dataset_root=DATASET_ROOT, cache_root=CACHE_ROOT, categories=tuple(CATEGORIES), purpose='nominal', batch_size=8, device=DEVICE, val_fraction=VAL_FRACTION, seed=SPLIT_SEED, image_size=224, mock_encoder=USE_MOCK_ENCODER, mock_embedding_dim=architecture['embedding_dim'], checkpoint=CHECKPOINT_PATH, official_repo=OFFICIAL_REPO_PATH)
cache_summary = run_cache_generation(cache_run)
assert cache_summary.ok, cache_summary.to_dict()
for item in cache_summary.results: print(f'{item.category}/{item.split}: {item.total} samples, encoded={item.encoded}, reused={item.skipped}')

## Train and save one best checkpoint per category

In [ ]:
import time
from ijepa_ad.caching.cache_schema import CacheExpectations, PreprocessingSpec
from ijepa_ad.caching.cli import MOCK_CACHE_SUBDIR, mock_checkpoint_sha256
from ijepa_ad.caching.latent_cache import compute_file_sha256
from ijepa_ad.data.transforms import PreprocessConfig
from ijepa_ad.predictor.cache_dataset import load_cached_latent_dataset
from ijepa_ad.predictor.transformer_predictor import LatentTransformerPredictor, PredictorConfig
from ijepa_ad.predictor.training import TrainingConfig, train_predictor
if USE_MOCK_ENCODER:
    cache_root, encoder_name, checkpoint_sha256 = CACHE_ROOT / MOCK_CACHE_SUBDIR, 'mock_encoder', mock_checkpoint_sha256(SPLIT_SEED)
else:
    cache_root, encoder_name, checkpoint_sha256 = CACHE_ROOT, 'ijepa_vit_g16_in22k_ep44', compute_file_sha256(CHECKPOINT_PATH)
preprocessing = PreprocessingSpec.from_config(PreprocessConfig())
results, durations = {}, {}
for category in CATEGORIES:
    def expected(split):
        return CacheExpectations(category=category, split=split, encoder_name=encoder_name, checkpoint_sha256=checkpoint_sha256, patch_size=16, embedding_dim=architecture['embedding_dim'], preprocessing=preprocessing)
    train_ds, val_ds = load_cached_latent_dataset(cache_root, expected('train')), load_cached_latent_dataset(cache_root, expected('val'))
    model_cfg = PredictorConfig(embedding_dim=train_ds.cache.metadata.embedding_dim, num_tokens=train_ds.cache.metadata.num_tokens, hidden_dim=architecture['hidden_dim'], depth=architecture['depth'], num_heads=architecture['num_heads'], mlp_ratio=architecture['mlp_ratio'], dropout=architecture['dropout'])
    checkpoint_path = OUTPUT_ROOT / f'{category}_{training["loss_type"]}.pt'
    started = time.perf_counter()
    result = train_predictor(LatentTransformerPredictor(model_cfg), train_ds, val_ds, TrainingConfig(**{**training, 'device': DEVICE}), checkpoint_path=checkpoint_path)
    durations[category] = time.perf_counter() - started
    results[category] = result
    print(f'{category}: best_epoch={result.best_epoch}, val_loss={result.best_val_loss:.6f}, checkpoint={result.checkpoint_path}')

In [ ]:
# Training report read back from the saved checkpoints (validation data only; no test metrics).
durations = globals().get('durations', {})
for category in CATEGORIES:
    path = OUTPUT_ROOT / f'{category}_{training["loss_type"]}.pt'
    payload = torch.load(path, map_location='cpu', weights_only=True)
    assert path.is_file() and payload['model_state_dict'] and payload['cache_provenance']
    history, best_epoch = payload['history'], payload['best_epoch']
    best = history[best_epoch - 1]
    assert best['epoch'] == best_epoch
    provenance, seconds = payload['cache_provenance'], durations.get(category)
    print(f'=== {category} ===')
    print(f'  checkpoint        : {path}  ({path.stat().st_size / 2**20:.2f} MiB)')
    print(f'  mask strategy     : {payload["training_config"]["mask_strategy"]}   loss: {payload["training_config"]["loss_type"]}')
    print(f'  training samples  : {len(provenance["train"]["sample_ids"])}   validation samples: {len(provenance["val"]["sample_ids"])}')
    print(f'  epochs run        : {len(history)} (max {payload["training_config"]["max_epochs"]})   best epoch: {best_epoch}')
    print(f'  train loss @ best : {best["train_loss"]:.6f}   last epoch: {history[-1]["train_loss"]:.6f}')
    print(f'  val loss @ best   : {payload["best_val_loss"]:.6f}')
    print(f'  training duration : ' + (f'{seconds:.1f} s (train_predictor call incl. checkpoint write)' if seconds is not None else 'n/a (training cell not run in this session)'))

Checkpoints are written to `checkpoints/predictors/<category>_cosine.pt`, the directory the UI and `scripts/evaluate_pilot.py` read by default. Payloads include model configuration, training configuration, loss history, selected epoch, and train/validation cache provenance. The report above covers training and validation only; test metrics come from `scripts/evaluate_pilot.py`. Load checkpoints with `load_predictor_checkpoint` for inference.